# Day 2

Today, we will start using nf-core pipelines to find differentially abundant genes in our dataset. 
We are using data from the following paper: https://www.nature.com/articles/s41593-023-01350-3#Sec10

1. Please take some time to read through the paper and understand their approach, hypotheses and goals.

What was the objective of the study?

What do the conditions mean?

oxy: oxycodone treatment


sal: saline placebo

What do the genotypes mean?

SNI: surgery where spared nerve injury was inflicted


Sham: surgery omitting the therapeutic component, performed in the single-blinded control group of experiments

Imagine you are the bioinformatician in the group who conducted this study. They hand you the raw files and ask you to analyze them.

What would you do?

Which groups would you compare to each other?
|              |  **SNI-Oxy**  |  **SNI-Sal**  |  **Sham-Oxy**  |  **Sham-Sal**  |
|--------------|:-------------:|:-------------:|:--------------:|:--------------:|
| **SNI-Oxy**  |               |      ✓        |        ✓      |        ✓       |
| **SNI-Sal**  |       ✓       |               |       𐄂       |        ✓       |
| **Sham-Oxy** |       ✓       |      𐄂        |               |        ✓       |
| **Sham-Sal** |       ✓       |       ✓       |       ✓       |                |


Please also mention which outcome you would expect to see from each comparison.

**Answer:**

Your group gave you a very suboptimal excel sheet (conditions_runs_oxy_project.xlsx) to get the information you need for each run they uploaded to the SRA.<br>
So, instead of directly diving into downloading the data and starting the analysis, you first need to sort the lazy table.<br>
Use Python and Pandas to get the table into a more sensible order.<br>

In [2]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import scipy.stats as statsy

In [47]:
data = pd.read_excel("conditions_runs_oxy_project.xlsx")

condition_map = {"condition: Sal": "Sal", "Condition: Oxy": "Oxy"}
genotype_map = {"Genotype: SNI": "SNI", "Genotype: Sham": "Sham"}

condition = data[list(condition_map)].notna().idxmax(axis=1).map(condition_map)
genotype = data[list(genotype_map)].notna().idxmax(axis=1).map(genotype_map)

combo = genotype + "-" + condition

new_data = data[["Patient", "Run", "RNA-seq", "DNA-seq"]].copy()

for label in combo.unique():
    new_data[label] = (combo == label).map({True: "x", False: pd.NA})

new_data

,Patient,Run,RNA-seq,DNA-seq,SNI-Sal,Sham-Oxy,Sham-Sal,SNI-Oxy
0,?,SRR23195505,x,NaN,x,NaN,NaN,NaN
1,?,SRR23195506,x,NaN,NaN,x,NaN,NaN
2,?,SRR23195507,x,NaN,NaN,NaN,x,NaN
3,?,SRR23195508,x,NaN,NaN,NaN,NaN,x
4,?,SRR23195509,x,NaN,NaN,NaN,NaN,x
5,?,SRR23195510,x,NaN,x,NaN,NaN,NaN
6,?,SRR23195511,x,NaN,NaN,x,NaN,NaN
7,?,SRR23195512,x,NaN,NaN,NaN,x,NaN
8,?,SRR23195513,x,NaN,x,NaN,NaN,NaN
9,?,SRR23195514,x,NaN,NaN,x,NaN,NaN


Then, perform some overview analysis and plot the results
1. How many samples do you have per condition?
2. How many samples do you have per genotype?
3. How often do you have each condition per genotype?

In [75]:
print(data['Condition: Oxy'].value_counts())
print(data['condition: Sal'].value_counts())
print(data['Genotype: SNI'].value_counts())
print(data['Genotype: Sham'].value_counts())
print(new_data['SNI-Oxy'].value_counts())
print(new_data['SNI-Sal'].value_counts())
print(new_data['Sham-Oxy'].value_counts())
print(new_data['Sham-Sal'].value_counts())


Condition: Oxy
x    8
Name: count, dtype: int64
condition: Sal
x    8
Name: count, dtype: int64
Genotype: SNI
x    8
Name: count, dtype: int64
Genotype: Sham
x    8
Name: count, dtype: int64
SNI-Oxy
x    4
Name: count, dtype: int64
SNI-Sal
x    4
Name: count, dtype: int64
Sham-Oxy
x    4
Name: count, dtype: int64
Sham-Sal
x    4
Name: count, dtype: int64


They were so kind to also provide you with the information of the number of bases per run, so that you can know how much space the data will take on your Cluster.<br>
Add a new column to your fancy table with this information (base_counts.csv) and sort your dataframe according to this information and the condition.

Then select the 2 smallest runs from your dataset and download them from SRA (maybe an nf-core pipeline can help here?...)

In [79]:
run_bases = pd.read_csv("base_counts.csv")
run_bases
new_data = new_data.merge(run_bases, how='left', on='Run')
new_data

,Patient,Run,RNA-seq,DNA-seq,SNI-Sal,Sham-Oxy,Sham-Sal,SNI-Oxy,Bases
0,?,SRR23195505,x,NaN,x,NaN,NaN,NaN,6922564500
1,?,SRR23195506,x,NaN,NaN,x,NaN,NaN,7859530800
2,?,SRR23195507,x,NaN,NaN,NaN,x,NaN,8063298900
3,?,SRR23195508,x,NaN,NaN,NaN,NaN,x,6927786900
4,?,SRR23195509,x,NaN,NaN,NaN,NaN,x,7003550100
5,?,SRR23195510,x,NaN,x,NaN,NaN,NaN,7377388500
6,?,SRR23195511,x,NaN,NaN,x,NaN,NaN,6456390900
7,?,SRR23195512,x,NaN,NaN,NaN,x,NaN,7462857900
8,?,SRR23195513,x,NaN,x,NaN,NaN,NaN,8099181600
9,?,SRR23195514,x,NaN,NaN,x,NaN,NaN,7226808600


In [85]:
new_data.nsmallest(2, "Bases").loc[:, ["Run", "Bases"]]

,Run,Bases
11,SRR23195516,6203117700
6,SRR23195511,6456390900


While your files are downloading, get back to the paper and explain how you would try to reproduce the analysis.<br>
When you are done with this shout, so we can discuss the different ideas.